# ERT data contracts and geometry

**Level 1 — Foundations**  
**Prerequisites:** Python basics; 35 min.

## Objectives

Trace the evidence from source contract to result. State units and assumptions, reproduce the calculation, and identify which inference the evidence cannot support.

## Setup

Uses repository modules and committed records through a discovered root; it downloads and mutates nothing. Install exact pins: `pip install numpy==2.2.6 matplotlib==3.11.2`

## Data

Provider-linked Slagdump source is documented, but no raw ERT file is present; upstream counts are metadata only.

## Method

A four-electrode datum uses current A/B and potential M/N. `R=ΔV/I` (Ω), `ρₐ=K R` (Ω·m), with `K=2π/(1/AM−1/AN−1/BM+1/BN)` in m for surface electrodes under a homogeneous half-space convention. Check sign and file-specific array ordering. The product parser preserves electrode coordinates, A-B-M-N indices and resistance; it does not infer absent columns. The source ledger describes the Slagdump example as provider-linked, but raw `.ohm` bytes are absent here and redistribution rights are not established. Thus no field pseudosection or inversion is fabricated. The executable diagnostic reports source readiness and plots only the committed mesh controls, clearly labelled as settings rather than geophysical measurements.

The geometric factor converts resistance under a half-space assumption; it is not true resistivity at a depth. Many cells affect each datum, creating broad sensitivity and depth-dependent resolution. Log parameterization protects positivity but changes smoothness meaning. Mesh refinement must be judged by stable predictions under comparable data fit, not cell size alone.

## Results

Read the physical axes and units; distinguish data fit, error against synthetic truth and agreement with analyst labels. Recomputed values must agree with the serialized receipt. A plotted model alone does not establish uniqueness, field transfer or resolution.

### Diagnostic 1

Inspect the committed case record and provenance.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Verify source ledger and local presence of raw ERT inputs.
import json
from pathlib import Path
ledger=json.loads((root/"data/source-ledger.json").read_text(encoding="utf-8")); print("ledger mentions Slagdump:","slagdump" in json.dumps(ledger).lower()); raw=list((root/"data").rglob("*.ohm")); print("local .ohm files:",raw,"; inversion from field arrays possible:",bool(raw))

### Diagnostic 2

Compute the declared comparison and plot it with physical units.

In [ ]:
# Read default and alternate mesh parameters from the source AST; no mesh or field model is generated.
import ast
import numpy as np
import matplotlib.pyplot as plt
tree=ast.parse((root/"data-pipeline/ert.py").read_text(encoding="utf-8")); assigns={n.targets[0].id:n.value for n in tree.body if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name)}
default=ast.literal_eval(assigns["INVERSE_OPTIONS"]); alt_node=assigns["ALTERNATE_MESH_OPTIONS"]; overrides={k:ast.literal_eval(v) for k,v in zip([k.value for k in alt_node.keys if k is not None],[v for v in alt_node.values if not isinstance(v,ast.Name)])}; alternate={**default,**overrides}
labels=["default","alternate"]; dx=[default["paraDX"],alternate["paraDX"]]; cell=[default["paraMaxCellSize"],alternate["paraMaxCellSize"]]; print("paraDX",dx,"dimensionless; maximum cell size",cell,"m; alternate/default ratios",cell[1]/cell[0],dx[1]/dx[0])
fig,ax=plt.subplots(1,2,figsize=(7,3.5),constrained_layout=True); x=np.arange(2); ax[0].bar(x,dx,color="#3274a1"); ax[0].set(xticks=x,xticklabels=labels,ylabel="paraDX (dimensionless)",title="Mesh refinement control"); ax[1].bar(x,cell,color="#e1812c"); ax[1].set(xticks=x,xticklabels=labels,ylabel="Maximum cell size (m)",title="Mesh cell size"); [a.grid(axis="y",alpha=.2) for a in ax]; fig.suptitle("ERT recipe settings only · no field result"); plt.show()

## Summary

Conclusion is conditional on geometry, acquisition, noise model, split, parameterization and source rights. Preserve negative cases and name the next observation or control needed.

## References

- [https://www.pygimli.org/user-guide/inversion/](https://www.pygimli.org/user-guide/inversion/)
- [https://www.pygimli.org/_examples_auto/3_ert/plot_02_ert_field_data/](https://www.pygimli.org/_examples_auto/3_ert/plot_02_ert_field_data/)